In [568]:
gen_report = True

In [569]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map({True: "yes", False: "no"})
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map({True: "yes", False: "no"})

df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"
df.loc[df["best_of_n"].isna(), "best_of_n"] = 1
df.loc[df["dataset_tt"].isna(), "dataset_tt"] = "one_shape"
df.loc[df["sampling_temperature_tt"].isna(), "sampling_temperature_tt"] = df["sampling_temperature"]

df.loc[(df["num_iterations"] == 0) | (df["best_of_n"] == 1), "test_time_mode"] = "-"

df.rename(columns={"test_time_training_accuracy": "test_time_mutual_accuracy"}, inplace=True)
df.rename(columns={"test_time_self_play_accuracy": "test_time_self_accuracy"}, inplace=True)

df["test_time_mode"] = df["test_time_mode"].replace("adaptation", "sample_adaptation")
df["dataset"] = df["dataset"].replace("shape", "shape1")
df["dataset_tt"] = df["dataset_tt"].replace("two_shape", "shape2")
df["dataset_tt"] = df["dataset_tt"].replace("shape_unique_double_attribute", "dual_attribute_shape")
# df["dataset_tt"] = df["dataset_tt"].replace("shape_unique_double_attribute", "single_attribute_shape")


df = df.where(pd.notna(df), "None")
pd.options.display.float_format = '{:.10g}'.format
df = df.map(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01) else x)
print("Total reports loaded:", len(df))


Total reports loaded: 387


/tmp/ipykernel_77470/2618864249.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
/tmp/ipykernel_77470/2618864249.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"


In [570]:
def filter_df(filters, df=df, sort_by=["message_length", "message_length_tt", "learning_rate_tt", "num_iterations"]):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=sort_by)
    

In [571]:
import base64

def to_html(df):    

    # ---- highlight rule ----
    # if 'mutual_play_accuracy' in df:
    #     max_col = 'mutual_play_accuracy'
    # else:
    #     max_col = 'test_accuracy'
    # max_val = pd.to_numeric(df[max_col]).max()

    # def highlight_max_row(row):
    #     if  pd.to_numeric(row[max_col]) == max_val:
    #         return ['font-weight: bold; background-color: #ffff99'] * len(row)
    #     else:
    #         return [''] * len(row)

    # ---- style ----
    styled = (
        df.style
            # .apply(highlight_max_row, axis=1)  # <<< APPLY HIGHLIGHT HERE
            .hide(axis="index")
            .format(
                lambda x: (
                    f"{int(x)}"                      # 20.0 -> 20
                    if isinstance(x, float) and x.is_integer()
                    else f"{x:.0e}"                  # small numbers -> scientific
                    if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01
                    else x
                )
            )
            .set_table_styles([
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {"selector": "th", "props": [
                    ("border-right", "1px solid black"),
                    ("color", "darkblue"),
                    ("font-weight", "bold"),
                    ("padding-left", "8px"),
                    ("padding-right", "8px")
                ]},
            ])
            .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)

        
def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = ["#0b3d91",  # dark blue
              "#800000",  # maroon
              "#205522",  # dark green
              "#111011",  # indigo
              "#444444"]  # dark gray
    color = colors[(num-1) % len(colors)]  # cycle through colors
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n')
        
def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")

def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')
        
        
def write(text):
    html_text = text.replace("\n", "<br>\n")
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,' +
        encoded +
        '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)
        
    if os.path.exists(path):
        os.remove(path)

In [572]:
import os
import shutil

def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [573]:
from itertools import product
import pandas as pd


def extract_maxes(df, cols=["seed", "message_length"], max_col="test_time_mutual_accuracy"):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[pd.to_numeric(section[max_col]).idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=cols)


def mean_and_std(df, out_cols=["VQEL", "dataset", "sim", "agent_a_training_mode"]):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i:i+3]

        mean = chunk["mutual_play_accuracy"].mean() * 100
        std = chunk["mutual_play_accuracy"].std() * 100

        rows.append({
            col: chunk[col].iloc[0] for col in out_cols} | {
            "mutual_play_accuracy": f"{mean:.1f} ± {std:.1f}"
        })

    out = pd.DataFrame(rows)
    return out.sort_values(by=out_cols)
    

In [574]:
clear()

---

In [575]:
backbone_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "test_time_mode",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "path",
]

backbone_cols_report = [
    "message_length",
    "agent_a_training_mode",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

baseline_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

baseline_cols_report = [
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

scaling_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

scaling_cols_report = [
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

adapt_cols = [
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

adapt_cols_report = [
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

# Shape - Euclidean

## Backbone

In [576]:

res = filter_df({
    "dataset": "shape1",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])
res = extract_maxes(res, cols=["agent_a_training_mode", "message_length"])

res[backbone_cols]


,VQEL,dataset,sim,message_length,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
3,True,shape1,euclidean,"[1, 2, 3, 4]",-,0.607,0.726,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...
0,True,shape1,euclidean,"[2, 3, 4]",-,0.664,0.789,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...
1,True,shape1,euclidean,"[3, 4]",-,0.752,0.798,20251221_0128_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape1,euclidean,[4],-,0.804,0.847,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...


## Baseline

In [577]:

res = filter_df({
    "dataset": "shape1",
    "sim": "euclidean",
    "dataset_tt": "shape2",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
336,True,shape1,euclidean,"[1, 2, 3, 4]",5,shape2,-,0.489,0.591,20251223_0952_bs32_vocab10_repr1024_lr1_0.001_...
354,True,shape1,euclidean,"[2, 3, 4]",5,shape2,-,0.53,0.591,20251223_0934_bs32_vocab10_repr1024_lr1_0.001_...
337,True,shape1,euclidean,[4],5,shape2,-,0.539,0.612,20251223_0935_bs32_vocab10_repr1024_lr1_0.001_...


## Scaling

In [578]:

res = filter_df({
    "dataset": "shape1",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_self_accuracy,test_time_mutual_accuracy,path
173,True,shape1,euclidean,"[1, 2, 3, 4]",5,shape2,scaling,1e-01,10,0.5,0.58,20251223_0139_bs32_vocab10_repr1024_lr1_0.001_...
18,True,shape1,euclidean,"[1, 2, 3, 4]",5,shape2,scaling,1e-01,20,0.503,0.58,20251223_0143_bs32_vocab10_repr1024_lr1_0.001_...
55,True,shape1,euclidean,"[1, 2, 3, 4]",5,shape2,scaling,1e-01,30,0.498,0.578,20251223_0152_bs32_vocab10_repr1024_lr1_0.001_...
323,True,shape1,euclidean,"[1, 2, 3, 4]",5,shape2,scaling,1e-02,10,0.492,0.591,20251223_0205_bs32_vocab10_repr1024_lr1_0.001_...
63,True,shape1,euclidean,"[1, 2, 3, 4]",5,shape2,scaling,1e-02,20,0.496,0.591,20251223_0209_bs32_vocab10_repr1024_lr1_0.001_...
248,True,shape1,euclidean,"[1, 2, 3, 4]",5,shape2,scaling,1e-02,30,0.495,0.592,20251223_0218_bs32_vocab10_repr1024_lr1_0.001_...
347,True,shape1,euclidean,"[2, 3, 4]",5,shape2,scaling,1e-01,10,0.546,0.586,20251223_0324_bs32_vocab10_repr1024_lr1_0.001_...
219,True,shape1,euclidean,"[2, 3, 4]",5,shape2,scaling,1e-01,20,0.545,0.592,20251223_0329_bs32_vocab10_repr1024_lr1_0.001_...
4,True,shape1,euclidean,"[2, 3, 4]",5,shape2,scaling,1e-01,30,0.545,0.587,20251223_0337_bs32_vocab10_repr1024_lr1_0.001_...
40,True,shape1,euclidean,"[2, 3, 4]",5,shape2,scaling,1e-02,10,0.532,0.59,20251223_0350_bs32_vocab10_repr1024_lr1_0.001_...


## Adaptation

In [579]:

res = filter_df({
    "dataset": "shape1",
    "sim": "euclidean",
    "VQEL": True,
    "test_time_mode": ["sample_adaptation", "batch_adaptation"]
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
286,True,shape1,euclidean,"[3, 4]",4,shape2,batch_adaptation,1e-05,200,0.536,0.555,20251227_2131_bs32_vocab10_repr1024_msg_len4_l...
377,True,shape1,euclidean,"[3, 4]",5,shape2,batch_adaptation,1e-05,200,0.54,0.567,20251227_2133_bs32_vocab10_repr1024_msg_len5_l...
368,True,shape1,euclidean,"[3, 4]",6,shape2,batch_adaptation,1e-05,200,0.543,0.572,20251227_2135_bs32_vocab10_repr1024_msg_len6_l...
318,True,shape1,euclidean,"[3, 4]",7,shape2,batch_adaptation,1e-05,200,0.524,0.554,20251227_2137_bs32_vocab10_repr1024_msg_len7_l...
327,True,shape1,euclidean,"[3, 4]",8,shape2,batch_adaptation,1e-05,200,0.538,0.556,20251227_2140_bs32_vocab10_repr1024_msg_len8_l...
51,True,shape1,euclidean,"[2, 3, 4]",5,shape2,sample_adaptation,1e-04,5,0.529,0.583,20251223_0415_bs32_vocab10_repr1024_lr1_0.001_...
53,True,shape1,euclidean,[4],5,shape2,sample_adaptation,1e-04,5,0.527,0.603,20251223_0601_bs32_vocab10_repr1024_lr1_0.001_...
271,True,shape1,euclidean,"[1, 2, 3, 4]",5,shape2,sample_adaptation,1e-04,5,0.503,0.579,20251223_0230_bs32_vocab10_repr1024_lr1_0.001_...
137,True,shape1,euclidean,[4],5,shape2,sample_adaptation,1e-04,10,0.525,0.579,20251223_0606_bs32_vocab10_repr1024_lr1_0.001_...
252,True,shape1,euclidean,"[2, 3, 4]",5,shape2,sample_adaptation,1e-04,10,0.524,0.571,20251223_0420_bs32_vocab10_repr1024_lr1_0.001_...


# Shape1

## Backbone

In [580]:

add_heading(2, "Shape1")
add_heading(3, "Base Model")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length"])
res = extract_maxes(res, cols=["message_length"])

to_html(res[backbone_cols_report])

res[backbone_cols]

,VQEL,dataset,sim,message_length,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
3,True,shape1,cosine,"[1, 2, 3, 4]",-,0.684,0.838,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...
0,True,shape1,cosine,"[2, 3, 4]",-,0.733,0.86,20251220_0820_bs32_vocab10_repr1024_lr1_0.0001...
1,True,shape1,cosine,"[3, 4]",-,0.841,0.889,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...
2,True,shape1,cosine,[4],-,0.857,0.907,20251220_1028_bs32_vocab10_repr1024_lr1_0.001_...


## Baseline

In [581]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "dataset_tt": "shape2",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": "-"
}, sort_by=["message_length_tt"])

to_html(res[baseline_cols_report])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
283,True,shape1,cosine,"[3, 4]",4,shape2,-,0.387,0.467,20251225_1828_bs32_vocab10_repr1024_lr1_0.0001...
386,True,shape1,cosine,"[3, 4]",5,shape2,-,0.414,0.473,20251223_0933_bs32_vocab10_repr1024_lr1_0.001_...
77,True,shape1,cosine,"[3, 4]",6,shape2,-,0.417,0.476,20251225_1829_bs32_vocab10_repr1024_lr1_0.0001...
189,True,shape1,cosine,"[3, 4]",7,shape2,-,0.414,0.448,20251225_1834_bs32_vocab10_repr1024_lr1_0.0001...
71,True,shape1,cosine,"[3, 4]",8,shape2,-,0.411,0.424,20251225_1830_bs32_vocab10_repr1024_lr1_0.0001...
378,True,shape1,cosine,"[3, 4]",9,shape2,-,0.406,0.409,20251225_1831_bs32_vocab10_repr1024_lr1_0.0001...


## Scaling

In [582]:
add_heading(3, "Scaling")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": "scaling"
}, sort_by=["message_length", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_self_accuracy,test_time_mutual_accuracy,path
344,True,shape1,cosine,"[3, 4]",5,shape2,scaling,1e-01,10,0.437,0.494,20251223_0017_bs32_vocab10_repr1024_lr1_0.001_...
84,True,shape1,cosine,"[3, 4]",5,shape2,scaling,1e-01,20,0.442,0.479,20251223_0021_bs32_vocab10_repr1024_lr1_0.001_...
290,True,shape1,cosine,"[3, 4]",5,shape2,scaling,1e-02,10,0.428,0.481,20251222_2351_bs32_vocab10_repr1024_lr1_0.001_...
167,True,shape1,cosine,"[3, 4]",5,shape2,scaling,1e-02,20,0.428,0.483,20251222_2356_bs32_vocab10_repr1024_lr1_0.001_...


## Adaptation

In [583]:
add_heading(3, "Adaptation")

res = filter_df({
    "dataset": "shape1",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": ["sample_adaptation", "batch_adaptation"],
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[adapt_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
199,True,shape1,cosine,"[3, 4]",4,shape2,batch_adaptation,1e-04,200,0.469,0.479,20251225_1807_bs32_vocab10_repr1024_lr1_0.0001...
356,True,shape1,cosine,"[3, 4]",4,shape2,batch_adaptation,1e-05,200,0.439,0.505,20251227_1456_bs32_vocab10_repr1024_msg_len4_l...
239,True,shape1,cosine,"[3, 4]",5,shape2,batch_adaptation,1e-04,200,0.509,0.521,20251225_1809_bs32_vocab10_repr1024_lr1_0.0001...
129,True,shape1,cosine,"[3, 4]",5,shape2,batch_adaptation,1e-05,200,0.463,0.517,20251227_1458_bs32_vocab10_repr1024_msg_len5_l...
160,True,shape1,cosine,"[3, 4]",6,shape2,batch_adaptation,1e-04,200,0.51,0.529,20251225_1812_bs32_vocab10_repr1024_lr1_0.0001...
121,True,shape1,cosine,"[3, 4]",6,shape2,batch_adaptation,1e-05,200,0.489,0.522,20251227_1500_bs32_vocab10_repr1024_msg_len6_l...
56,True,shape1,cosine,"[3, 4]",7,shape2,batch_adaptation,1e-04,200,0.53,0.536,20251225_1814_bs32_vocab10_repr1024_lr1_0.0001...
22,True,shape1,cosine,"[3, 4]",7,shape2,batch_adaptation,1e-05,200,0.483,0.519,20251227_1503_bs32_vocab10_repr1024_msg_len7_l...
93,True,shape1,cosine,"[3, 4]",8,shape2,batch_adaptation,1e-04,200,0.534,0.533,20251225_1817_bs32_vocab10_repr1024_lr1_0.0001...
195,True,shape1,cosine,"[3, 4]",8,shape2,batch_adaptation,1e-05,200,0.482,0.505,20251227_1505_bs32_vocab10_repr1024_msg_len8_l...


In [584]:
add_heading(4, "Max")
final = extract_maxes(res)
to_html(final[adapt_cols_report])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,True,shape1,cosine,"[3, 4]",9,shape2,batch_adaptation,1e-04,200,0.552,0.541,20251225_1819_bs32_vocab10_repr1024_lr1_0.0001...


# Shape12

## Backbone

In [585]:
add_heading(2, "Shape12")
add_heading(3, "Base Model")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "VQEL": True,
    "dataset_tt": "shape12",
    "test_time_mode": "-"
}, sort_by=["message_length"])

to_html(res[backbone_cols_report])

res[backbone_cols]

,VQEL,dataset,sim,message_length,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
169,True,shape12,cosine,"[3, 4]",-,0.771,0.863,20251225_1907_bs32_vocab10_repr1024_lr1_0.0001...


## Baseline

In [586]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "dataset_tt": "shape3",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": "-"
}, sort_by=["message_length_tt"])

to_html(res[baseline_cols_report])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
152,True,shape12,cosine,"[3, 4]",4,shape3,-,0.59,0.664,20251225_2040_bs32_vocab10_repr1024_lr1_0.0001...
192,True,shape12,cosine,"[3, 4]",5,shape3,-,0.65,0.732,20251225_2050_bs32_vocab10_repr1024_lr1_0.0001...
282,True,shape12,cosine,"[3, 4]",6,shape3,-,0.616,0.737,20251225_2052_bs32_vocab10_repr1024_lr1_0.0001...
72,True,shape12,cosine,"[3, 4]",7,shape3,-,0.51,0.715,20251225_2054_bs32_vocab10_repr1024_lr1_0.0001...
91,True,shape12,cosine,"[3, 4]",8,shape3,-,0.336,0.658,20251225_2055_bs32_vocab10_repr1024_lr1_0.0001...
369,True,shape12,cosine,"[3, 4]",9,shape3,-,0.183,0.601,20251225_2056_bs32_vocab10_repr1024_lr1_0.0001...
322,True,shape12,cosine,"[3, 4]",10,shape3,-,0.087,0.569,20251225_2103_bs32_vocab10_repr1024_lr1_0.0001...


## Adaptation

In [587]:
add_heading(3, "Adaptation")

res = filter_df({
    "dataset": "shape12",
    "sim": "cosine",
    "VQEL": True,
    "message_length": "[3, 4]",
    "test_time_mode": ["sample_adaptation", "batch_adaptation"],
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[adapt_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
30,True,shape12,cosine,"[3, 4]",4,shape3,batch_adaptation,1e-04,200,0.567,0.507,20251225_1939_bs32_vocab10_repr1024_lr1_0.0001...
329,True,shape12,cosine,"[3, 4]",4,shape3,batch_adaptation,1e-05,200,0.661,0.684,20251227_0228_bs32_vocab10_repr1024_lr1_0.0001...
245,True,shape12,cosine,"[3, 4]",5,shape3,batch_adaptation,1e-04,200,0.674,0.616,20251225_1941_bs32_vocab10_repr1024_lr1_0.0001...
34,True,shape12,cosine,"[3, 4]",5,shape3,batch_adaptation,1e-05,200,0.747,0.766,20251227_0230_bs32_vocab10_repr1024_lr1_0.0001...
200,True,shape12,cosine,"[3, 4]",6,shape3,batch_adaptation,1e-04,200,0.778,0.713,20251225_1944_bs32_vocab10_repr1024_lr1_0.0001...
98,True,shape12,cosine,"[3, 4]",6,shape3,batch_adaptation,1e-05,200,0.796,0.827,20251227_0233_bs32_vocab10_repr1024_lr1_0.0001...
361,True,shape12,cosine,"[3, 4]",7,shape3,batch_adaptation,1e-04,200,0.773,0.706,20251225_1946_bs32_vocab10_repr1024_lr1_0.0001...
196,True,shape12,cosine,"[3, 4]",7,shape3,batch_adaptation,1e-05,200,0.741,0.841,20251227_0235_bs32_vocab10_repr1024_lr1_0.0001...
143,True,shape12,cosine,"[3, 4]",8,shape3,batch_adaptation,1e-04,200,0.775,0.595,20251225_1949_bs32_vocab10_repr1024_lr1_0.0001...
372,True,shape12,cosine,"[3, 4]",8,shape3,batch_adaptation,1e-05,200,0.628,0.793,20251227_0238_bs32_vocab10_repr1024_lr1_0.0001...


In [588]:
add_heading(4, "Max")
final = extract_maxes(res)
to_html(final[adapt_cols_report])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,True,shape12,cosine,"[3, 4]",7,shape3,batch_adaptation,1e-05,200,0.741,0.841,20251227_0235_bs32_vocab10_repr1024_lr1_0.0001...


# MNIST

## Backbone

In [589]:
add_heading(2, "MNIST")
add_heading(3, 'Base Model')
res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-",
    "dataset_tt": "mnist1"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[backbone_cols_report])

res[backbone_cols]

,VQEL,dataset,sim,message_length,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
47,True,mnist1,cosine,"[1, 2, 3, 4]",-,0.76,0.893,20251225_0503_bs32_vocab10_repr192_lr1_0.0001_...
371,True,mnist1,cosine,"[2, 3, 4]",-,0.794,0.911,20251225_0543_bs32_vocab10_repr192_lr1_0.0001_...
82,True,mnist1,cosine,"[3, 4]",-,0.788,0.895,20251223_1750_bs32_vocab10_repr192_lr1_0.0001_...
39,True,mnist1,cosine,[4],-,0.905,0.908,20251225_0626_bs32_vocab10_repr192_lr1_0.0001_...


## Baseline

In [590]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "dataset_tt": "mnist2",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[baseline_cols_report])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
95,True,mnist1,cosine,"[3, 4]",4,mnist2,-,0.464,0.439,20251224_1857_bs32_vocab10_repr192_lr1_0.0001_...
80,True,mnist1,cosine,"[3, 4]",5,mnist2,-,0.509,0.481,20251225_1756_bs32_vocab10_repr192_lr1_0.0001_...
41,True,mnist1,cosine,"[3, 4]",6,mnist2,-,0.534,0.5,20251223_1959_bs32_vocab10_repr192_lr1_0.0001_...
353,True,mnist1,cosine,"[3, 4]",7,mnist2,-,0.527,0.503,20251225_1757_bs32_vocab10_repr192_lr1_0.0001_...
67,True,mnist1,cosine,"[3, 4]",8,mnist2,-,0.518,0.495,20251224_0942_bs32_vocab10_repr192_lr1_0.0001_...
164,True,mnist1,cosine,"[3, 4]",9,mnist2,-,0.49,0.475,20251225_1758_bs32_vocab10_repr192_lr1_0.0001_...
99,True,mnist1,cosine,"[3, 4]",10,mnist2,-,0.46,0.436,20251224_1856_bs32_vocab10_repr192_lr1_0.0001_...


## Scaling

In [591]:
add_heading(3, "Scaling")
res = filter_df({
    "dataset": "mnist1",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "scaling"
}, sort_by=["message_length", "message_length_tt", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_self_accuracy,test_time_mutual_accuracy,path
205,True,mnist1,cosine,"[3, 4]",4,mnist2,scaling,1e-01,100,1e-02,1e-02,20251224_1329_bs32_vocab1_repr192_lr1_0.0001_l...
274,True,mnist1,cosine,"[3, 4]",4,mnist2,scaling,1e-02,100,1e-02,1e-02,20251224_1256_bs32_vocab1_repr192_lr1_0.0001_l...
130,True,mnist1,cosine,"[3, 4]",5,mnist2,scaling,1e-02,100,0.529,0.494,20251224_2337_bs32_vocab10_repr192_lr1_0.0001_...
258,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-01,20,0.561,0.502,20251223_1933_bs32_vocab10_repr192_lr1_0.0001_...
70,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-01,30,0.559,0.49,20251223_1943_bs32_vocab10_repr192_lr1_0.0001_...
76,True,mnist1,cosine,"[3, 4]",6,mnist2,scaling,1e-02,100,0.553,0.518,20251225_0016_bs32_vocab10_repr192_lr1_0.0001_...
208,True,mnist1,cosine,"[3, 4]",8,mnist2,scaling,1e-01,100,0.59,0.508,20251223_2255_bs32_vocab10_repr192_lr1_0.0001_...
109,True,mnist1,cosine,"[3, 4]",8,mnist2,scaling,1e-02,100,0.545,0.524,20251223_2157_bs32_vocab10_repr192_lr1_0.0001_...
94,True,mnist1,cosine,"[3, 4]",10,mnist2,scaling,1e-01,100,0.56,0.45,20251224_1144_bs32_vocab10_repr192_lr1_0.0001_...
85,True,mnist1,cosine,"[3, 4]",10,mnist2,scaling,1e-02,100,0.511,0.474,20251224_1034_bs32_vocab10_repr192_lr1_0.0001_...


## Adaptation

In [592]:
add_heading(3, "Adaptation")

res = filter_df({
    "dataset": "mnist1",
    "VQEL": True,
    "test_time_mode": ["sample_adaptation", "batch_adaptation"]
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[adapt_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
112,True,mnist1,cosine,"[3, 4]",4,mnist2,batch_adaptation,1e-04,200,0.536,0.472,20251225_1528_bs32_vocab10_repr192_lr1_0.0001_...
174,True,mnist1,cosine,"[3, 4]",5,mnist2,batch_adaptation,1e-04,200,0.605,0.54,20251225_1530_bs32_vocab10_repr192_lr1_0.0001_...
114,True,mnist1,cosine,"[3, 4]",6,mnist2,batch_adaptation,1e-04,200,0.664,0.589,20251225_1531_bs32_vocab10_repr192_lr1_0.0001_...
352,True,mnist1,cosine,"[3, 4]",7,mnist2,batch_adaptation,1e-04,200,0.686,0.604,20251225_1533_bs32_vocab10_repr192_lr1_0.0001_...
74,True,mnist1,cosine,"[3, 4]",8,mnist2,batch_adaptation,1e-04,200,0.705,0.617,20251225_1535_bs32_vocab10_repr192_lr1_0.0001_...
46,True,mnist1,cosine,"[3, 4]",9,mnist2,batch_adaptation,1e-04,200,0.719,0.62,20251225_1538_bs32_vocab10_repr192_lr1_0.0001_...
175,True,mnist1,cosine,"[3, 4]",10,mnist2,batch_adaptation,1e-04,200,0.731,0.609,20251225_1540_bs32_vocab10_repr192_lr1_0.0001_...
146,True,mnist1,cosine,"[3, 4]",4,mnist2,sample_adaptation,1e-04,5,0.475,0.444,20251224_1440_bs32_vocab10_repr192_lr1_0.0001_...
123,True,mnist1,cosine,"[3, 4]",4,mnist2,sample_adaptation,1e-04,10,0.456,0.419,20251224_1444_bs32_vocab10_repr192_lr1_0.0001_...
289,True,mnist1,cosine,"[3, 4]",4,mnist2,sample_adaptation,1e-04,15,0.427,0.386,20251224_1450_bs32_vocab10_repr192_lr1_0.0001_...


In [593]:
add_heading(4, "Max")
final = extract_maxes(res)
to_html(final[adapt_cols_report])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,True,mnist1,cosine,"[3, 4]",9,mnist2,batch_adaptation,1e-04,200,0.719,0.62,20251225_1538_bs32_vocab10_repr192_lr1_0.0001_...


# ImageNet

## Backbone

In [594]:
add_heading(2, "ImageNet")
add_heading(3, "Base Model")

res = filter_df({
    "dataset": "imagenet",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-",
    "dataset_tt": "imagenet"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[backbone_cols_report])

res[backbone_cols]

,VQEL,dataset,sim,message_length,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
359,True,imagenet,cosine,"[3, 4]",-,0.895,0.893,20251225_1135_bs32_vocab10_repr2048_lr1_0.0001...


## Baseline

In [595]:
add_heading(3, "Baseline")
res = filter_df({
    "dataset": "imagenet",
    "dataset_tt": "imagenet_same_class",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[baseline_cols_report])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
247,True,imagenet,cosine,"[3, 4]",4,imagenet_same_class,-,0.437,0.429,20251225_1737_bs32_vocab10_repr2048_lr1_0.0001...
306,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,-,0.441,0.416,20251225_1739_bs32_vocab10_repr2048_lr1_0.0001...
19,True,imagenet,cosine,"[3, 4]",6,imagenet_same_class,-,0.352,0.262,20251225_1745_bs32_vocab10_repr2048_lr1_0.0001...
342,True,imagenet,cosine,"[3, 4]",7,imagenet_same_class,-,0.221,0.116,20251225_1740_bs32_vocab10_repr2048_lr1_0.0001...
257,True,imagenet,cosine,"[3, 4]",8,imagenet_same_class,-,0.101,0.098,20251225_1746_bs32_vocab10_repr2048_lr1_0.0001...
246,True,imagenet,cosine,"[3, 4]",9,imagenet_same_class,-,0.043,0.091,20251225_1741_bs32_vocab10_repr2048_lr1_0.0001...
156,True,imagenet,cosine,"[3, 4]",10,imagenet_same_class,-,0.025,0.092,20251225_1748_bs32_vocab10_repr2048_lr1_0.0001...


## Scaling

In [596]:
add_heading(3, "Scaling")
res = filter_df({
    "dataset": "imagenet",
    "VQEL": True,
    "test_time_mode": 'scaling'
}, sort_by=["message_length_tt", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_self_accuracy,test_time_mutual_accuracy,path
3,True,imagenet,cosine,"[3, 4]",4,imagenet_same_class,scaling,1e-01,200,0.424,0.364,20251227_1718_bs32_vocab10_repr2048_msg_len4_l...
135,True,imagenet,cosine,"[3, 4]",4,imagenet_same_class,scaling,1e-02,100,0.448,0.448,20251227_1511_bs32_vocab10_repr2048_msg_len4_l...
75,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,scaling,1e-01,200,0.474,0.403,20251227_1805_bs32_vocab10_repr2048_msg_len5_l...
297,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,scaling,1e-02,100,0.438,0.411,20251227_1535_bs32_vocab10_repr2048_msg_len5_l...
104,True,imagenet,cosine,"[3, 4]",6,imagenet_same_class,scaling,1e-01,200,0.419,0.29,20251227_1903_bs32_vocab10_repr2048_msg_len6_l...
153,True,imagenet,cosine,"[3, 4]",6,imagenet_same_class,scaling,1e-02,100,0.404,0.287,20251227_1604_bs32_vocab10_repr2048_msg_len6_l...
259,True,imagenet,cosine,"[3, 4]",7,imagenet_same_class,scaling,1e-01,200,0.388,0.233,20251227_2012_bs32_vocab10_repr2048_msg_len7_l...
279,True,imagenet,cosine,"[3, 4]",7,imagenet_same_class,scaling,1e-02,100,0.257,0.161,20251227_1638_bs32_vocab10_repr2048_msg_len7_l...


## Adaptation

In [597]:
add_heading(3, "Adaptation")
res = filter_df({
    "dataset": "imagenet",
    "VQEL": True,
    "test_time_mode": ["sample_adaptation", "batch_adaptation"]
}, sort_by=["test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[adapt_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
374,True,imagenet,cosine,"[3, 4]",4,imagenet_same_class,batch_adaptation,1e-04,200,0.667,0.579,20251225_1455_bs32_vocab10_repr2048_lr1_0.0001...
338,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,batch_adaptation,1e-04,200,0.718,0.623,20251225_1458_bs32_vocab10_repr2048_lr1_0.0001...
275,True,imagenet,cosine,"[3, 4]",6,imagenet_same_class,batch_adaptation,1e-04,200,0.765,0.6,20251225_1501_bs32_vocab10_repr2048_lr1_0.0001...
227,True,imagenet,cosine,"[3, 4]",7,imagenet_same_class,batch_adaptation,1e-04,200,0.752,0.537,20251225_1506_bs32_vocab10_repr2048_lr1_0.0001...
118,True,imagenet,cosine,"[3, 4]",8,imagenet_same_class,batch_adaptation,1e-04,200,0.721,0.521,20251225_1511_bs32_vocab10_repr2048_lr1_0.0001...
321,True,imagenet,cosine,"[3, 4]",9,imagenet_same_class,batch_adaptation,1e-04,200,0.665,0.268,20251225_1516_bs32_vocab10_repr2048_lr1_0.0001...
363,True,imagenet,cosine,"[3, 4]",10,imagenet_same_class,batch_adaptation,1e-04,200,0.591,0.171,20251225_1522_bs32_vocab10_repr2048_lr1_0.0001...
346,True,imagenet,cosine,"[3, 4]",4,imagenet_same_class,sample_adaptation,1e-04,200,0.266,0.232,20251225_2329_bs32_vocab10_repr2048_lr1_0.0001...
110,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,sample_adaptation,1e-04,200,0.311,0.281,20251226_0102_bs32_vocab10_repr2048_lr1_0.0001...
317,True,imagenet,cosine,"[3, 4]",6,imagenet_same_class,sample_adaptation,1e-04,200,0.288,0.262,20251226_0251_bs32_vocab10_repr2048_lr1_0.0001...


In [598]:
add_heading(4, "Max")
final = extract_maxes(res)
to_html(final[adapt_cols_report])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
0,True,imagenet,cosine,"[3, 4]",5,imagenet_same_class,batch_adaptation,1e-04,200,0.718,0.623,20251225_1458_bs32_vocab10_repr2048_lr1_0.0001...


# Shape Single Attribute  

## Backbone

In [599]:
add_heading(2, "Single Attribute  Shape")
add_heading(3, "Base Model")
res = filter_df({
    "dataset": "shape_unique_single_attribute",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-",
    "dataset_tt": "shape_unique_single_attribute"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[backbone_cols_report])

res[backbone_cols]

,VQEL,dataset,sim,message_length,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
345,True,shape_unique_single_attribute,cosine,"[1, 2]",-,0.642,0.678,20251227_0118_bs1_vocab10_repr1024_lr1_0.0001_...
255,True,shape_unique_single_attribute,cosine,"[2, 3]",-,0.82,0.863,20251227_0101_bs1_vocab10_repr1024_lr1_0.0001_...
58,True,shape_unique_single_attribute,cosine,[2],-,0.711,0.721,20251227_0132_bs1_vocab10_repr1024_lr1_0.0001_...
302,True,shape_unique_single_attribute,cosine,"[3, 4]",-,0.93,0.941,20251226_1816_bs1_vocab10_repr1024_lr1_0.0001_...
240,True,shape_unique_single_attribute,cosine,[4],-,0.957,0.947,20251227_0148_bs1_vocab10_repr1024_lr1_0.0001_...


## Baseline

In [600]:
add_heading(3, "Baseline")
res = filter_df({
    "dataset": "shape_unique_single_attribute",
    "dataset_tt": "dual_attribute_shape",
    "VQEL": True,
    "test_time_mode": "-"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[baseline_cols_report])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
168,True,shape_unique_single_attribute,cosine,"[1, 2]",2,dual_attribute_shape,-,0.314,0.379,20251227_1325_bs1_vocab10_repr1024_msg_len2_lr...
170,True,shape_unique_single_attribute,cosine,"[1, 2]",3,dual_attribute_shape,-,0.482,0.51,20251227_1326_bs1_vocab10_repr1024_msg_len3_lr...
35,True,shape_unique_single_attribute,cosine,"[1, 2]",4,dual_attribute_shape,-,0.531,0.477,20251227_1326_bs1_vocab10_repr1024_msg_len4_lr...
132,True,shape_unique_single_attribute,cosine,"[1, 2]",5,dual_attribute_shape,-,0.581,0.45,20251227_1327_bs1_vocab10_repr1024_msg_len5_lr...
194,True,shape_unique_single_attribute,cosine,"[1, 2]",6,dual_attribute_shape,-,0.585,0.376,20251227_1327_bs1_vocab10_repr1024_msg_len6_lr...
333,True,shape_unique_single_attribute,cosine,"[2, 3]",3,dual_attribute_shape,-,0.597,0.634,20251227_1328_bs1_vocab10_repr1024_msg_len3_lr...
249,True,shape_unique_single_attribute,cosine,"[2, 3]",4,dual_attribute_shape,-,0.696,0.74,20251227_1328_bs1_vocab10_repr1024_msg_len4_lr...
127,True,shape_unique_single_attribute,cosine,"[2, 3]",5,dual_attribute_shape,-,0.721,0.716,20251227_1329_bs1_vocab10_repr1024_msg_len5_lr...
358,True,shape_unique_single_attribute,cosine,"[2, 3]",6,dual_attribute_shape,-,0.716,0.602,20251227_1329_bs1_vocab10_repr1024_msg_len6_lr...
15,True,shape_unique_single_attribute,cosine,"[2, 3]",7,dual_attribute_shape,-,0.712,0.491,20251227_1330_bs1_vocab10_repr1024_msg_len7_lr...


In [601]:
add_heading(4, "Max")
final = extract_maxes(res)
to_html(final[baseline_cols_report])
final[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
3,True,shape_unique_single_attribute,cosine,"[1, 2]",3,dual_attribute_shape,-,0.482,0.51,20251227_1326_bs1_vocab10_repr1024_msg_len3_lr...
0,True,shape_unique_single_attribute,cosine,"[2, 3]",4,dual_attribute_shape,-,0.696,0.74,20251227_1328_bs1_vocab10_repr1024_msg_len4_lr...
1,True,shape_unique_single_attribute,cosine,[2],2,dual_attribute_shape,-,0.415,0.449,20251227_1333_bs1_vocab10_repr1024_msg_len2_lr...
2,True,shape_unique_single_attribute,cosine,"[3, 4]",5,dual_attribute_shape,-,0.845,0.865,20251226_1843_bs1_vocab10_repr1024_lr1_0.0001_...
4,True,shape_unique_single_attribute,cosine,[4],4,dual_attribute_shape,-,0.85,0.853,20251227_1333_bs1_vocab10_repr1024_msg_len4_lr...


## Scaling

In [602]:
add_heading(3, "Scaling")
res = filter_df({
    "dataset": "shape_unique_single_attribute",
    "VQEL": True,
    "test_time_mode": 'scaling'
}, sort_by=["message_length_tt", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_self_accuracy,test_time_mutual_accuracy,path


## Adaptation

In [603]:
add_heading(3, "Adaptation")
res = filter_df({
    "dataset": "shape_unique_single_attribute",
    "VQEL": True,
    "test_time_mode": ["sample_adaptation", "batch_adaptation"]
}, sort_by=["message_length", "test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[adapt_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
213,True,shape_unique_single_attribute,cosine,"[1, 2]",2,dual_attribute_shape,batch_adaptation,1e-05,200,0.347,0.355,20251227_1158_bs1_vocab10_repr1024_msg_len2_lr...
177,True,shape_unique_single_attribute,cosine,"[1, 2]",2,dual_attribute_shape,batch_adaptation,1e-05,300,0.342,0.341,20251227_1430_bs1_vocab10_repr1024_msg_len2_lr...
214,True,shape_unique_single_attribute,cosine,"[1, 2]",3,dual_attribute_shape,batch_adaptation,1e-05,200,0.547,0.48,20251227_1200_bs1_vocab10_repr1024_msg_len3_lr...
172,True,shape_unique_single_attribute,cosine,"[1, 2]",3,dual_attribute_shape,batch_adaptation,1e-05,300,0.547,0.47,20251227_1432_bs1_vocab10_repr1024_msg_len3_lr...
117,True,shape_unique_single_attribute,cosine,"[1, 2]",4,dual_attribute_shape,batch_adaptation,1e-05,200,0.645,0.501,20251227_1201_bs1_vocab10_repr1024_msg_len4_lr...
54,True,shape_unique_single_attribute,cosine,"[1, 2]",4,dual_attribute_shape,batch_adaptation,1e-05,300,0.67,0.505,20251227_1435_bs1_vocab10_repr1024_msg_len4_lr...
25,True,shape_unique_single_attribute,cosine,"[1, 2]",5,dual_attribute_shape,batch_adaptation,1e-05,200,0.732,0.491,20251227_1203_bs1_vocab10_repr1024_msg_len5_lr...
314,True,shape_unique_single_attribute,cosine,"[1, 2]",5,dual_attribute_shape,batch_adaptation,1e-05,300,0.746,0.479,20251227_1438_bs1_vocab10_repr1024_msg_len5_lr...
147,True,shape_unique_single_attribute,cosine,"[1, 2]",6,dual_attribute_shape,batch_adaptation,1e-05,200,0.739,0.448,20251227_1206_bs1_vocab10_repr1024_msg_len6_lr...
100,True,shape_unique_single_attribute,cosine,"[1, 2]",6,dual_attribute_shape,batch_adaptation,1e-05,300,0.791,0.432,20251227_1441_bs1_vocab10_repr1024_msg_len6_lr...


In [604]:
add_heading(4, "Max")
final = extract_maxes(res)
to_html(final[adapt_cols_report])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
3,True,shape_unique_single_attribute,cosine,"[1, 2]",4,dual_attribute_shape,batch_adaptation,1e-05,300,0.67,0.505,20251227_1435_bs1_vocab10_repr1024_msg_len4_lr...
0,True,shape_unique_single_attribute,cosine,"[2, 3]",4,dual_attribute_shape,batch_adaptation,1e-05,200,0.766,0.719,20251227_1210_bs1_vocab10_repr1024_msg_len4_lr...
1,True,shape_unique_single_attribute,cosine,[2],2,dual_attribute_shape,batch_adaptation,1e-05,200,0.423,0.422,20251227_1228_bs1_vocab10_repr1024_msg_len2_lr...
2,True,shape_unique_single_attribute,cosine,"[3, 4]",5,dual_attribute_shape,batch_adaptation,1e-05,200,0.879,0.894,20251227_0212_bs1_vocab10_repr1024_lr1_0.0001_...
4,True,shape_unique_single_attribute,cosine,[4],4,dual_attribute_shape,batch_adaptation,1e-05,200,0.885,0.849,20251227_1229_bs1_vocab10_repr1024_msg_len4_lr...


# Single- or Dual-Attribute Shape


## Backbone

In [605]:
add_heading(2, "Single- or Dual-Attribute Shape")
add_heading(3, "Base Model")
res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "sim": "cosine",
    "VQEL": True,
    "test_time_mode": "-",
    "pretrained_checkpoint_a": "None"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[backbone_cols_report])

res[backbone_cols]

,VQEL,dataset,sim,message_length,test_time_mode,self_play_accuracy_a,mutual_play_accuracy,path
7,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",-,0.618,0.645,20251227_0009_bs1_vocab10_repr1024_lr1_0.0001_...
11,True,shape_unique_atleast_one_attribute,cosine,"[2, 3]",-,0.861,0.863,20251226_2352_bs1_vocab10_repr1024_lr1_0.0001_...
134,True,shape_unique_atleast_one_attribute,cosine,[2],-,0.709,0.714,20251227_0023_bs1_vocab10_repr1024_lr1_0.0001_...
180,True,shape_unique_atleast_one_attribute,cosine,"[3, 4]",-,0.944,0.938,20251226_1853_bs1_vocab10_repr1024_lr1_0.0001_...
24,True,shape_unique_atleast_one_attribute,cosine,[4],-,0.961,0.953,20251227_0039_bs1_vocab10_repr1024_lr1_0.0001_...


## Baseline

In [606]:
add_heading(3, "Baseline")
res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "VQEL": True,
    "test_time_mode": "-",
    "pretrained_checkpoint_a": "!None"
}, sort_by=["message_length", "message_length_tt"])

to_html(res[baseline_cols_report])

res[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
270,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",2,dual_attribute_shape,-,0.32,0.358,20251227_1042_bs1_vocab10_repr1024_lr1_0.0001_...
31,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",3,dual_attribute_shape,-,0.447,0.474,20251227_1045_bs1_vocab10_repr1024_msg_len3_lr...
304,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",4,dual_attribute_shape,-,0.499,0.479,20251227_1043_bs1_vocab10_repr1024_lr1_0.0001_...
102,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",5,dual_attribute_shape,-,0.555,0.459,20251227_1045_bs1_vocab10_repr1024_msg_len5_lr...
265,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",6,dual_attribute_shape,-,0.557,0.399,20251227_1046_bs1_vocab10_repr1024_msg_len6_lr...
384,True,shape_unique_atleast_one_attribute,cosine,"[2, 3]",3,dual_attribute_shape,-,0.59,0.621,20251227_0948_bs1_vocab10_repr1024_lr1_0.0001_...
43,True,shape_unique_atleast_one_attribute,cosine,"[2, 3]",4,dual_attribute_shape,-,0.704,0.74,20251227_0949_bs1_vocab10_repr1024_lr1_0.0001_...
29,True,shape_unique_atleast_one_attribute,cosine,"[2, 3]",5,dual_attribute_shape,-,0.664,0.702,20251227_0950_bs1_vocab10_repr1024_lr1_0.0001_...
37,True,shape_unique_atleast_one_attribute,cosine,[2],2,dual_attribute_shape,-,0.416,0.409,20251227_0945_bs1_vocab10_repr1024_lr1_0.0001_...
326,True,shape_unique_atleast_one_attribute,cosine,[2],3,dual_attribute_shape,-,0.09,0.139,20251227_0947_bs1_vocab10_repr1024_lr1_0.0001_...


In [607]:
add_heading(4, "Max")
final = extract_maxes(res)
to_html(final[baseline_cols_report])
final[baseline_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,test_time_self_accuracy,test_time_mutual_accuracy,path
3,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",4,dual_attribute_shape,-,0.499,0.479,20251227_1043_bs1_vocab10_repr1024_lr1_0.0001_...
0,True,shape_unique_atleast_one_attribute,cosine,"[2, 3]",4,dual_attribute_shape,-,0.704,0.74,20251227_0949_bs1_vocab10_repr1024_lr1_0.0001_...
1,True,shape_unique_atleast_one_attribute,cosine,[2],2,dual_attribute_shape,-,0.416,0.409,20251227_0945_bs1_vocab10_repr1024_lr1_0.0001_...
2,True,shape_unique_atleast_one_attribute,cosine,"[3, 4]",5,dual_attribute_shape,-,0.848,0.832,20251226_2120_bs1_vocab10_repr1024_lr1_0.0001_...
4,True,shape_unique_atleast_one_attribute,cosine,[4],4,dual_attribute_shape,-,0.867,0.86,20251227_0943_bs1_vocab10_repr1024_lr1_0.0001_...


## Scaling

In [608]:
add_heading(3, "Scaling")
res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "VQEL": True,
    "test_time_mode": 'scaling'
}, sort_by=["message_length_tt", "sampling_temperature_tt", "best_of_n"])

to_html(res[scaling_cols_report])

res[scaling_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,sampling_temperature_tt,best_of_n,test_time_self_accuracy,test_time_mutual_accuracy,path


## Adaptation

In [609]:
add_heading(3, "Adaptation")
res = filter_df({
    "dataset": "shape_unique_atleast_one_attribute",
    "VQEL": True,
    "test_time_mode": ["sample_adaptation", "batch_adaptation"]
}, sort_by=["message_length", "test_time_mode", "message_length_tt", "learning_rate_tt", "num_iterations"])

to_html(res[adapt_cols_report])

res[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
113,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",2,dual_attribute_shape,batch_adaptation,1e-05,200,0.325,0.317,20251227_1056_bs1_vocab10_repr1024_msg_len2_lr...
228,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",3,dual_attribute_shape,batch_adaptation,1e-05,200,0.441,0.449,20251227_1047_bs1_vocab10_repr1024_msg_len3_lr...
202,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",4,dual_attribute_shape,batch_adaptation,1e-05,200,0.559,0.495,20251227_1049_bs1_vocab10_repr1024_msg_len4_lr...
254,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",4,dual_attribute_shape,batch_adaptation,1e-05,300,0.582,0.514,20251227_1405_bs1_vocab10_repr1024_msg_len4_lr...
198,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",4,dual_attribute_shape,batch_adaptation,1e-05,400,0.604,0.505,20251227_1352_bs1_vocab10_repr1024_msg_len4_lr...
48,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",5,dual_attribute_shape,batch_adaptation,1e-05,200,0.61,0.521,20251227_1051_bs1_vocab10_repr1024_msg_len5_lr...
181,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",5,dual_attribute_shape,batch_adaptation,1e-05,300,0.656,0.519,20251227_1408_bs1_vocab10_repr1024_msg_len5_lr...
382,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",5,dual_attribute_shape,batch_adaptation,1e-05,400,0.675,0.53,20251227_1356_bs1_vocab10_repr1024_msg_len5_lr...
38,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",6,dual_attribute_shape,batch_adaptation,1e-05,200,0.66,0.508,20251227_1053_bs1_vocab10_repr1024_msg_len6_lr...
366,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",6,dual_attribute_shape,batch_adaptation,1e-05,400,0.741,0.481,20251227_1400_bs1_vocab10_repr1024_msg_len6_lr...


In [610]:
add_heading(4, "Max")
final = extract_maxes(res)
to_html(final[adapt_cols_report])
final[adapt_cols]

,VQEL,dataset,sim,message_length,message_length_tt,dataset_tt,test_time_mode,learning_rate_tt,num_iterations,test_time_self_accuracy,test_time_mutual_accuracy,path
3,True,shape_unique_atleast_one_attribute,cosine,"[1, 2]",5,dual_attribute_shape,batch_adaptation,1e-05,400,0.675,0.53,20251227_1356_bs1_vocab10_repr1024_msg_len5_lr...
0,True,shape_unique_atleast_one_attribute,cosine,"[2, 3]",4,dual_attribute_shape,batch_adaptation,1e-05,200,0.767,0.709,20251227_1127_bs1_vocab10_repr1024_msg_len4_lr...
1,True,shape_unique_atleast_one_attribute,cosine,[2],2,dual_attribute_shape,batch_adaptation,1e-05,200,0.411,0.397,20251227_1144_bs1_vocab10_repr1024_msg_len2_lr...
2,True,shape_unique_atleast_one_attribute,cosine,"[3, 4]",6,dual_attribute_shape,batch_adaptation,1e-05,300,0.934,0.894,20251226_2152_bs1_vocab10_repr1024_lr1_0.0001_...
4,True,shape_unique_atleast_one_attribute,cosine,[4],4,dual_attribute_shape,batch_adaptation,1e-05,200,0.889,0.852,20251227_1148_bs1_vocab10_repr1024_msg_len4_lr...
